This notebook is used to add toxicity prediction to the top 20 recomended molecules

For antiviral and antibacterial drug development, these toxicity endpoints are particularly critical:

Essential Acute Toxicity Endpoints:
- Hepatotoxicity - Liver toxicity is one of the most common reasons drugs fail. The liver metabolizes most drugs, and anti-infectives often require high doses or prolonged treatment. Check for ALT/AST elevation predictions.
- Cardiotoxicity (hERG inhibition) - Blocking the hERG potassium channel can cause fatal cardiac arrhythmias. This is a major regulatory concern and frequent cause of drug withdrawal.
- Nephrotoxicity - Kidney toxicity is especially important since many drugs are renally cleared and infections can already stress kidney function.

Genotoxicity/Mutagenicity:
- AMES test prediction - Identifies potential DNA-damaging effects that could lead to cancer. Regulatory agencies require this data.
- Chromosome aberration - Another genotoxicity concern for long-term safety.
- Specific Considerations for Anti-infectives:
- Mitochondrial toxicity - Particularly important for antivirals since some (like older NRTIs) have caused mitochondrial dysfunction. Can lead to lactic acidosis, liver failure.
- Bone marrow suppression/Hematotoxicity - Many anti-infectives can suppress blood cell production, leading to anemia, neutropenia, or thrombocytopenia.
- Cytotoxicity against human cells - You want selectivity for pathogen over host cells. High general cytotoxicity is a red flag.

Drug-Drug Interaction Potential:
- CYP450 inhibition/induction - Patients with infections often take multiple medications. Strong CYP interactions can cause toxicity or reduce efficacy of other drugs.

In [1]:
import sys
import os

# Completely suppress stderr output
sys.stderr = open(os.devnull, 'w')

# Now import everything
import warnings
warnings.filterwarnings('ignore')

2025-11-04 12:51:10.026025: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-04 12:51:11.098408: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-04 12:51:15.943448: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-04 13:02:26.

In [2]:
import numpy as np
import pandas as pd
import pubchempy as pcp

In [3]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
modelBuildingDataDir = os.path.join(dataDir, 'modelBuildingData/')
resultsDir = os.path.join(dataDir, 'Results/')

In [4]:
EnamineAntiviralsData_top20 = pd.read_csv(resultsDir + "virus/EnamineAntiviralsCompounds_allVirus.csv")
EnamineAntiviralsData_top20

,Rank,SMILES,pPotency_prediction,IC50 (M)
0,1,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,7.089,8.149146e-08
1,2,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,6.840,1.445621e-07
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,6.794,1.608349e-07
3,4,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,6.650,2.238847e-07
4,5,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,6.643,2.273610e-07
5,6,CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC,6.629,2.347164e-07
6,7,CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C,6.600,2.510905e-07
7,8,CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O,6.573,2.674945e-07
8,9,CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C,6.547,2.837417e-07
9,10,COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2,6.529,2.959537e-07


### Get Pubchem ID's

In [35]:
def get_pubchem_cid(smiles):
    """Get PubChem CID from SMILES"""
    try:
        compounds = pcp.get_compounds(smiles, 'smiles')
        if compounds:
            return compounds[0].cid
        else:
            return None
    except:
        return None

# Add CID column to your dataframe
EnamineAntiviralsData_top20['PubChem_CID'] = EnamineAntiviralsData_top20['SMILES'].apply(get_pubchem_cid)
EnamineAntiviralsData_top20

,Rank,SMILES,pPotency_prediction,IC50 (M),PubChem_CID
0,1,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,7.089,8.149146e-08,3316217
1,2,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,6.840,1.445621e-07,53528130
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,6.794,1.608349e-07,71898137
3,4,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,6.650,2.238847e-07,3123513
4,5,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,6.643,2.273610e-07,47052959
5,6,CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC,6.629,2.347164e-07,16856689
6,7,CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C,6.600,2.510905e-07,16854878
7,8,CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O,6.573,2.674945e-07,75507949
8,9,CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C,6.547,2.837417e-07,45833555
9,10,COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2,6.529,2.959537e-07,47049980


### Use RDKit for basic druglikeness

In [36]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# Calculate Lipinski's Rule of Five
def check_lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    return mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10

# Apply to your dataframe
EnamineAntiviralsData_top20['Lipinski_Pass'] = EnamineAntiviralsData_top20['SMILES'].apply(check_lipinski)
EnamineAntiviralsData_top20

,Rank,SMILES,pPotency_prediction,IC50 (M),PubChem_CID,Lipinski_Pass
0,1,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,7.089,8.149146e-08,3316217,True
1,2,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,6.840,1.445621e-07,53528130,True
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,6.794,1.608349e-07,71898137,True
3,4,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,6.650,2.238847e-07,3123513,True
4,5,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,6.643,2.273610e-07,47052959,True
5,6,CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC,6.629,2.347164e-07,16856689,True
6,7,CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C,6.600,2.510905e-07,16854878,True
7,8,CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O,6.573,2.674945e-07,75507949,True
8,9,CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C,6.547,2.837417e-07,45833555,True
9,10,COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2,6.529,2.959537e-07,47049980,True


### Find toxicity data from different data bases

### 1. Using ADMET_AI

In [37]:
from admet_ai import ADMETModel

# Initialize model
model = ADMETModel()

# Get predictions for all SMILES
smiles_list = EnamineAntiviralsData_top20['SMILES'].tolist()
predictions = model.predict(smiles=smiles_list)

# Check what predictions looks like
print("Predictions type:", type(predictions))
print("Predictions shape:", predictions.shape if hasattr(predictions, 'shape') else len(predictions))

# Reset indices before concatenating
predictions_reset = predictions.reset_index(drop=True)
original_reset = EnamineAntiviralsData_top20.reset_index(drop=True)

# Concatenate with reset indices
EnamineAntiviralsData_top20 = pd.concat([original_reset, predictions_reset], axis=1)
EnamineAntiviralsData_top20.head()

Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.bias".
Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.b

,Rank,SMILES,pPotency_prediction,IC50 (M),PubChem_CID,Lipinski_Pass,molecular_weight,logP,hydrogen_bond_acceptors,hydrogen_bond_donors,...,Caco2_Wang_drugbank_approved_percentile,Clearance_Hepatocyte_AZ_drugbank_approved_percentile,Clearance_Microsome_AZ_drugbank_approved_percentile,Half_Life_Obach_drugbank_approved_percentile,HydrationFreeEnergy_FreeSolv_drugbank_approved_percentile,LD50_Zhu_drugbank_approved_percentile,Lipophilicity_AstraZeneca_drugbank_approved_percentile,PPBR_AZ_drugbank_approved_percentile,Solubility_AqSolDB_drugbank_approved_percentile,VDss_Lombardo_drugbank_approved_percentile
0,1,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,7.089,8.149146e-08,3316217,True,343.387,0.70610,7,1,...,70.453664,72.857697,61.031408,32.531989,26.289259,63.086468,52.811167,56.417216,34.625824,26.134161
1,2,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,6.840,1.445621e-07,53528130,True,332.360,0.31830,6,2,...,41.372625,59.092672,28.576968,8.724312,5.583560,34.082978,29.042264,32.376890,43.117487,51.841799
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,6.794,1.608349e-07,71898137,True,318.417,2.67160,4,1,...,97.324544,89.220628,80.069794,55.874370,58.898798,59.247770,75.222955,45.715394,39.666537,16.518030
3,4,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,6.650,2.238847e-07,3123513,True,342.359,0.51570,8,2,...,45.560295,54.672354,39.744087,38.037999,12.330361,46.452113,45.986817,54.749903,23.768903,18.844513
4,5,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,6.643,2.273610e-07,47052959,True,312.369,3.09552,5,1,...,73.284219,52.151997,70.802637,19.426134,28.887166,45.133773,59.674292,58.200853,49.360217,11.050795


### Check what are the columns has been added by ADMET

In [38]:
# Check all new columns added by ADMET_AI
#print(EnamineAntiviralsData_top20.columns.tolist())

admet_cols = [col for col in EnamineAntiviralsData_top20.columns 
              if col not in ['Rank', 'SMILES', 'pPotency_prediction', 'IC50 (M)', 'PubChem_CID', 'Lipinski_Pass']]
print(f"\nADMET_AI added {len(admet_cols)} columns:")
print(admet_cols)


ADMET_AI added 98 columns:
['molecular_weight', 'logP', 'hydrogen_bond_acceptors', 'hydrogen_bond_donors', 'Lipinski', 'QED', 'stereo_centers', 'tpsa', 'AMES', 'BBB_Martins', 'Bioavailability_Ma', 'CYP1A2_Veith', 'CYP2C19_Veith', 'CYP2C9_Substrate_CarbonMangels', 'CYP2C9_Veith', 'CYP2D6_Substrate_CarbonMangels', 'CYP2D6_Veith', 'CYP3A4_Substrate_CarbonMangels', 'CYP3A4_Veith', 'Carcinogens_Lagunin', 'ClinTox', 'DILI', 'HIA_Hou', 'NR-AR-LBD', 'NR-AR', 'NR-AhR', 'NR-Aromatase', 'NR-ER-LBD', 'NR-ER', 'NR-PPAR-gamma', 'PAMPA_NCATS', 'Pgp_Broccatelli', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'Skin_Reaction', 'hERG', 'Caco2_Wang', 'Clearance_Hepatocyte_AZ', 'Clearance_Microsome_AZ', 'Half_Life_Obach', 'HydrationFreeEnergy_FreeSolv', 'LD50_Zhu', 'Lipophilicity_AstraZeneca', 'PPBR_AZ', 'Solubility_AqSolDB', 'VDss_Lombardo', 'molecular_weight_drugbank_approved_percentile', 'logP_drugbank_approved_percentile', 'hydrogen_bond_acceptors_drugbank_approved_percentile', 'hydrogen_bond_do

### Define critical columns for anti-infectives

In [39]:
critical_toxicity = [
    'AMES',              # Mutagenicity - CRITICAL
    'hERG',              # Cardiotoxicity - CRITICAL  
    'DILI',              # Liver injury - CRITICAL
    'ClinTox',           # Clinical toxicity failures
    'Carcinogens_Lagunin',  # Carcinogenicity
    'LD50_Zhu'           # Acute toxicity (lower = more toxic)
]

cyp_inhibition = [
    'CYP1A2_Veith',      # Drug-drug interactions
    'CYP2C19_Veith',
    'CYP2C9_Veith', 
    'CYP2D6_Veith',
    'CYP3A4_Veith'       # Most important - metabolizes 50% of drugs
]

adme_properties = [
    'Bioavailability_Ma',           # Oral availability
    'HIA_Hou',                      # Intestinal absorption
    'Solubility_AqSolDB',           # Water solubility
    'Caco2_Wang',                   # Permeability
    'Clearance_Hepatocyte_AZ',      # Hepatic clearance
    'Half_Life_Obach',              # Elimination half-life
    'PPBR_AZ',                      # Plasma protein binding
    'Pgp_Broccatelli'               # Efflux pump substrate
]

# Create focused dataframe with key columns
key_columns = ['Rank', 'SMILES', 'IC50 (M)', 'pPotency_prediction'] + \
              critical_toxicity + cyp_inhibition + adme_properties

EnamineAntiviralsData_wToxicity_clean = EnamineAntiviralsData_top20[key_columns].copy()

print("Key ADMET predictions:")
EnamineAntiviralsData_wToxicity_clean.head()

Key ADMET predictions:


,Rank,SMILES,IC50 (M),pPotency_prediction,AMES,hERG,DILI,ClinTox,Carcinogens_Lagunin,LD50_Zhu,...,CYP2D6_Veith,CYP3A4_Veith,Bioavailability_Ma,HIA_Hou,Solubility_AqSolDB,Caco2_Wang,Clearance_Hepatocyte_AZ,Half_Life_Obach,PPBR_AZ,Pgp_Broccatelli
0,1,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,8.149146e-08,7.089,0.357161,0.779022,0.771195,0.430878,0.185121,2.723966,...,0.117314,0.077892,0.868528,0.999009,-3.849915,-4.652979,63.715802,-2.171157,79.850997,0.385678
1,2,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,1.445621e-07,6.840,0.366818,0.280867,0.784952,0.195772,0.209623,2.275151,...,0.006129,0.053598,0.839785,0.996997,-3.382136,-5.120838,46.249348,-19.475079,61.020120,0.046939
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,1.608349e-07,6.794,0.312303,0.349044,0.260252,0.228934,0.160226,2.661411,...,0.056462,0.540797,0.856946,0.999968,-3.566324,-4.092612,96.697039,11.094887,71.754358,0.104327
3,4,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,2.238847e-07,6.650,0.604157,0.737543,0.967874,0.339789,0.145750,2.478743,...,0.032536,0.067904,0.911981,0.999458,-4.412575,-5.052590,40.988078,0.537885,78.615040,0.351507
4,5,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,2.273610e-07,6.643,0.280445,0.851717,0.372983,0.063424,0.059125,2.459117,...,0.788197,0.903310,0.898522,0.999969,-3.069134,-4.609028,38.190232,-9.814090,81.380915,0.736024


### Calculate toxicity risk scores:

In [40]:
def calculate_toxicity_risk(row):
    """
    Calculate risk scores for each molecule
    Returns individual risks and overall score
    """
    risks = {}
    
    # Critical toxicity flags (binary: 1 = toxic, 0 = safe)
    # For ADMET_AI, values > 0.5 typically indicate positive prediction
    risks['mutagenic'] = 1 if row['AMES'] > 0.5 else 0
    risks['cardiotoxic'] = 1 if row['hERG'] > 0.5 else 0
    risks['hepatotoxic'] = 1 if row['DILI'] > 0.5 else 0
    risks['clinical_tox'] = 1 if row['ClinTox'] > 0.5 else 0
    risks['carcinogenic'] = 1 if row['Carcinogens_Lagunin'] > 0.5 else 0
    
    # LD50: lower values = more toxic (in mol/kg)
    # Flag if predicted LD50 < 50 mg/kg (convert from mol/kg)
    risks['acute_tox'] = 1 if row['LD50_Zhu'] < 0.0002 else 0  # ~50 mg/kg for MW~250
    
    # CYP inhibition count (drug-drug interaction risk)
    cyp_inhibitions = sum([
        row['CYP1A2_Veith'] > 0.5,
        row['CYP2C19_Veith'] > 0.5,
        row['CYP2C9_Veith'] > 0.5,
        row['CYP2D6_Veith'] > 0.5,
        row['CYP3A4_Veith'] > 0.5
    ])
    risks['cyp_inhibitor_count'] = cyp_inhibitions
    risks['high_ddi_risk'] = 1 if cyp_inhibitions >= 2 else 0
    
    # Overall toxicity score (0 = best, higher = worse)
    risks['total_tox_flags'] = (risks['mutagenic'] + risks['cardiotoxic'] + 
                                risks['hepatotoxic'] + risks['clinical_tox'] + 
                                risks['carcinogenic'] + risks['acute_tox'])
    
    # Safety classification
    if risks['total_tox_flags'] == 0:
        risks['safety_class'] = 'Low Risk'
    elif risks['total_tox_flags'] <= 1:
        risks['safety_class'] = 'Moderate Risk'
    else:
        risks['safety_class'] = 'High Risk'
    
    return pd.Series(risks)

# Apply risk scoring
risk_scores = EnamineAntiviralsData_wToxicity_clean.apply(calculate_toxicity_risk, axis=1)
EnamineAntiviralsData_wToxicity_clean = pd.concat([EnamineAntiviralsData_wToxicity_clean, risk_scores], axis=1)
EnamineAntiviralsData_wToxicity_clean

,Rank,SMILES,IC50 (M),pPotency_prediction,AMES,hERG,DILI,ClinTox,Carcinogens_Lagunin,LD50_Zhu,...,mutagenic,cardiotoxic,hepatotoxic,clinical_tox,carcinogenic,acute_tox,cyp_inhibitor_count,high_ddi_risk,total_tox_flags,safety_class
0,1,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,8.149146e-08,7.089,0.357161,0.779022,0.771195,0.430878,0.185121,2.723966,...,0,1,1,0,0,0,0,0,2,High Risk
1,2,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,1.445621e-07,6.840,0.366818,0.280867,0.784952,0.195772,0.209623,2.275151,...,0,0,1,0,0,0,0,0,1,Moderate Risk
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,1.608349e-07,6.794,0.312303,0.349044,0.260252,0.228934,0.160226,2.661411,...,0,0,0,0,0,0,2,1,0,Low Risk
3,4,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,2.238847e-07,6.650,0.604157,0.737543,0.967874,0.339789,0.145750,2.478743,...,1,1,1,0,0,0,1,0,3,High Risk
4,5,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,2.273610e-07,6.643,0.280445,0.851717,0.372983,0.063424,0.059125,2.459117,...,0,1,0,0,0,0,4,1,1,Moderate Risk
5,6,CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC,2.347164e-07,6.629,0.340428,0.533507,0.880549,0.248680,0.105339,2.608681,...,0,1,1,0,0,0,0,0,2,High Risk
6,7,CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C,2.510905e-07,6.600,0.245955,0.606687,0.846172,0.336997,0.093442,2.611802,...,0,1,1,0,0,0,0,0,2,High Risk
7,8,CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O,2.674945e-07,6.573,0.304816,0.372344,0.500742,0.492966,0.226198,2.794022,...,0,0,1,0,0,0,1,0,1,Moderate Risk
8,9,CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C,2.837417e-07,6.547,0.137849,0.296734,0.651543,0.344248,0.089953,2.457961,...,0,0,1,0,0,0,0,0,1,Moderate Risk
9,10,COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2,2.959537e-07,6.529,0.371379,0.139196,0.582636,0.211225,0.039807,2.384435,...,0,0,1,0,0,0,0,0,1,Moderate Risk


### Toxicity profile

### Evaluate drug-like ADME properties

In [42]:
def evaluate_adme(row):
    """
    Evaluate drug-like ADME properties
    """
    adme_scores = {}
    
    # Good oral bioavailability (want high, >0.5 is good)
    adme_scores['good_bioavailability'] = 1 if row['Bioavailability_Ma'] > 0.5 else 0
    
    # Good intestinal absorption (want high)
    adme_scores['good_absorption'] = 1 if row['HIA_Hou'] > 0.5 else 0
    
    # Good solubility (higher is better, usually log scale)
    adme_scores['good_solubility'] = 1 if row['Solubility_AqSolDB'] > -4 else 0
    
    # Good permeability (Caco2 > 8 nm/s is good)
    adme_scores['good_permeability'] = 1 if row['Caco2_Wang'] > 8 else 0
    
    # Reasonable half-life (want 3-12 hours for most drugs)
    adme_scores['good_half_life'] = 1 if 3 < row['Half_Life_Obach'] < 12 else 0
    
    # Not a strong Pgp substrate (want low, <0.5)
    adme_scores['low_efflux'] = 1 if row['Pgp_Broccatelli'] < 0.5 else 0
    
    # ADME score (0-6, higher is better)
    adme_scores['adme_score'] = sum([
        adme_scores['good_bioavailability'],
        adme_scores['good_absorption'],
        adme_scores['good_solubility'],
        adme_scores['good_permeability'],
        adme_scores['good_half_life'],
        adme_scores['low_efflux']
    ])
    
    # Classification
    if adme_scores['adme_score'] >= 5:
        adme_scores['adme_class'] = 'Excellent'
    elif adme_scores['adme_score'] >= 3:
        adme_scores['adme_class'] = 'Good'
    else:
        adme_scores['adme_class'] = 'Poor'
    
    return pd.Series(adme_scores)

# Apply ADME evaluation
adme_eval = EnamineAntiviralsData_wToxicity_clean.apply(evaluate_adme, axis=1)
EnamineAntiviralsData_wToxicity_wADME = pd.concat([EnamineAntiviralsData_wToxicity_clean, adme_eval], axis=1)
EnamineAntiviralsData_wToxicity_wADME = EnamineAntiviralsData_wToxicity_wADME.filter(
    items=['Rank', 'SMILES', 'pPotency_prediction', 'IC50 (M)',
           'good_bioavailability', 'good_absorption', 'good_solubility', 'adme_score', 'adme_class', 'total_tox_flags']
)
EnamineAntiviralsData_wToxicity_wADME

,Rank,SMILES,pPotency_prediction,IC50 (M),good_bioavailability,good_absorption,good_solubility,adme_score,adme_class,total_tox_flags
0,1,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,7.089,8.149146e-08,1,1,1,4,Good,2
1,2,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,6.840,1.445621e-07,1,1,1,4,Good,1
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,6.794,1.608349e-07,1,1,1,5,Excellent,0
3,4,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,6.650,2.238847e-07,1,1,0,3,Good,3
4,5,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,6.643,2.273610e-07,1,1,1,3,Good,1
5,6,CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC,6.629,2.347164e-07,1,1,1,4,Good,2
6,7,CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C,6.600,2.510905e-07,1,1,1,4,Good,2
7,8,CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O,6.573,2.674945e-07,1,1,1,4,Good,1
8,9,CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C,6.547,2.837417e-07,1,1,1,5,Excellent,1
9,10,COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2,6.529,2.959537e-07,1,1,1,4,Good,1


### Create overall ranking combining potency, safety, and ADME

### Identify best drug candidates

In [45]:
EnamineBestCandidates = EnamineAntiviralsData_wToxicity_wADME[
    (EnamineAntiviralsData_wToxicity_wADME['total_tox_flags'] == 0) &  # No major toxicity flags
    (EnamineAntiviralsData_wToxicity_wADME['adme_score'] >= 3) &        # Decent ADME
    (EnamineAntiviralsData_wToxicity_wADME['IC50 (M)'] < 1e-5)         # Good potency (< 10 µM)
]
EnamineBestCandidates['IC50 (M)'] = EnamineBestCandidates['IC50 (M)'].apply(lambda x: f"{x:.3e}")

print(f"Found {len(EnamineBestCandidates)} compounds with:")
print("  No toxicity flags")
print("  Good ADME properties (score ≥ 3)")
print("  IC50 < 10 µM")
EnamineBestCandidates

Found 5 compounds with:
  No toxicity flags
  Good ADME properties (score ≥ 3)
  IC50 < 10 µM


,Rank,SMILES,pPotency_prediction,IC50 (M),good_bioavailability,good_absorption,good_solubility,adme_score,adme_class,total_tox_flags
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,6.794,1.608e-07,1,1,1,5,Excellent,0
10,11,CC(NC(=O)NC1=CC=CN=C1N2CCCCC2)C(=O)NC(C)(C)C,6.522,3.007e-07,1,1,1,4,Good,0
14,15,COC=1N=CC=CC1NC(=O)NCCC(=O)N2CCCC2,6.421,3.793e-07,1,1,1,4,Good,0
16,17,CC(C)=CC1C(C(=O)NC=2C=CN(CC(N)=O)N2)C1(C)C,6.356,4.406e-07,1,1,1,4,Good,0
19,20,CC(C)=CC1C(C(=O)NCCC=2C=CC=CN2)C1(C)C,6.307,4.935e-07,1,1,1,4,Good,0


### 2. Using DeepChem for Tox21 Predictions

2.1 Use DeepChem's built-in Tox21 predictions

In [10]:
import deepchem as dc
import pandas as pd
import numpy as np
from rdkit import Chem

def predict_tox21_manual(smiles_list):
    """
    Predict Tox21 without using load_tox21 (which has bugs)
    """
    # Download and load Tox21 data directly
    import urllib.request
    import os
    
    print("Downloading Tox21 dataset...")
    toxicitydataDir = dataDir + '/tox21_data'
    os.makedirs(toxicitydataDir, exist_ok=True)
    
    dataset_file = os.path.join(toxicitydataDir, 'tox21.csv')
    
    if not os.path.exists(dataset_file):
        url = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz'
        urllib.request.urlretrieve(url, dataset_file + '.gz')
        import gzip
        import shutil
        with gzip.open(dataset_file + '.gz', 'rb') as f_in:
            with open(dataset_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
    
    # Load the CSV
    print("Loading Tox21 data from CSV...")
    df = pd.read_csv(dataset_file)
    
    # Tox21 tasks
    tasks = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 
             'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 
             'SR-HSE', 'SR-MMP', 'SR-p53']
    
    print(f"Dataset shape: {df.shape}")
    print(f"Tasks: {tasks}")
    
    # Featurize training data
    print("Featurizing training data...")
    featurizer = dc.feat.CircularFingerprint(size=1024)
    
    # Initialize lists to collect data
    train_features = []
    train_labels = []
    
    for idx, row in df.iterrows():
        try:
            smiles = row['smiles']
            feat = featurizer.featurize([smiles])[0]
            
            # Check if featurization was successful
            if feat is not None and len(feat) == 1024:  # Ensure correct size
                # Convert to 1D array if needed
                feat_array = np.array(feat).flatten()
                
                if not np.isnan(feat_array).any():
                    train_features.append(feat_array)
                    # Get labels for all tasks
                    labels = [float(row[task]) if pd.notna(row[task]) else 0.0 for task in tasks]
                    train_labels.append(labels)
                
        except Exception as e:
            continue
        
        if (idx + 1) % 1000 == 0:
            print(f"Processed {idx + 1}/{len(df)} molecules...")
    
    print(f"Successfully featurized {len(train_features)} training molecules")
    
    # Convert to arrays - stack them properly
    X_train = np.vstack(train_features)  # Use vstack instead of array
    y_train = np.array(train_labels)
    
    print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")
    
    # Create dataset
    train_dataset = dc.data.NumpyDataset(X=X_train, y=y_train)
    
    # Train model - FIXED VERSION
    print("Training model...")
    
    # Use sklearn-based model instead (more stable)
    from sklearn.ensemble import RandomForestClassifier
    
    # Train a separate model for each task
    models = []
    for task_idx, task_name in enumerate(tasks):
        print(f"  Training {task_name}...")
        y_task = y_train[:, task_idx]
        
        # Only train on samples with valid labels
        valid_mask = ~np.isnan(y_task)
        X_valid = X_train[valid_mask]
        y_valid = y_task[valid_mask].astype(int)
        
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_valid, y_valid)
        models.append(model)
    
    # Now predict on your molecules
    print(f"\nFeaturizing {len(smiles_list)} query molecules...")
    query_features = []
    valid_query_indices = []
    
    for i, smiles in enumerate(smiles_list):
        try:
            feat = featurizer.featurize([smiles])[0]
            if feat is not None and len(feat) == 1024:
                feat_array = np.array(feat).flatten()
                if not np.isnan(feat_array).any():
                    query_features.append(feat_array)
                    valid_query_indices.append(i)
        except:
            continue
    
    print(f"Successfully featurized {len(query_features)}/{len(smiles_list)} query molecules")
    
    # Predict
    X_query = np.vstack(query_features)  # Use vstack
    
    print("Making predictions...")
    predictions = []
    for task_idx, model in enumerate(models):
        # Get probability of positive class
        pred_proba = model.predict_proba(X_query)[:, 1]
        predictions.append(pred_proba)
    
    # Transpose to get (n_samples, n_tasks) shape
    predictions = np.array(predictions).T
    
    # Create results DataFrame
    results = pd.DataFrame(predictions, columns=tasks, index=valid_query_indices)
    
    return results, tasks

# Run
print("Starting Tox21 predictions with DeepChem...\n")
EnamineAntiviralsData_wToxicity_deepchem, tox21_tasks = predict_tox21_manual(
    EnamineAntiviralsData_top20['SMILES'].tolist()
)

print("\nDeepChem Tox21 predictions:")
EnamineAntiviralsData_wToxicity_deepchem.head()

Starting Tox21 predictions with DeepChem...

Loading Tox21 data from CSV...
Dataset shape: (7831, 14)
Tasks: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']
Featurizing training data...
Processed 1000/7831 molecules...
Processed 2000/7831 molecules...
Processed 3000/7831 molecules...
Processed 4000/7831 molecules...
Processed 5000/7831 molecules...
Processed 6000/7831 molecules...
Processed 7000/7831 molecules...
Successfully featurized 7823 training molecules
Training data shape: X=(7823, 1024), y=(7823, 12)
Training model...
  Training NR-AR...
  Training NR-AR-LBD...
  Training NR-AhR...
  Training NR-Aromatase...
  Training NR-ER...
  Training NR-ER-LBD...
  Training NR-PPAR-gamma...
  Training SR-ARE...
  Training SR-ATAD5...
  Training SR-HSE...
  Training SR-MMP...
  Training SR-p53...

Featurizing 20 query molecules...
Successfully featurized 20/20 query molecules
Making predictions...



,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53
0,0.020714,0.010514,0.136735,0.023666,0.089574,0.037979,0.017535,0.141416,0.066652,0.045342,0.089071,0.052564
1,0.019247,0.010541,0.094948,0.034347,0.094115,0.023413,0.030239,0.125769,0.050595,0.037296,0.078907,0.037930
2,0.051570,0.033400,0.116305,0.025556,0.087873,0.019591,0.011712,0.108590,0.028621,0.065245,0.117644,0.039300
3,0.041308,0.021898,0.163825,0.033119,0.110762,0.024316,0.028378,0.132531,0.095349,0.045216,0.091601,0.052118
4,0.017548,0.012191,0.186113,0.061238,0.098721,0.051221,0.015372,0.103082,0.040477,0.057636,0.124210,0.048687


In [14]:
# Add to original dataframe
EnamineAntiviralsData_wToxicity_deepchem_final = EnamineAntiviralsData_top20.copy()
for task in tox21_tasks:
    col_name = f'DC_{task}'
    EnamineAntiviralsData_wToxicity_deepchem_final[col_name] = np.nan
    EnamineAntiviralsData_wToxicity_deepchem_final.loc[EnamineAntiviralsData_wToxicity_deepchem.index, col_name] = \
        EnamineAntiviralsData_wToxicity_deepchem[task].values

print("\n DeepChem-compatible predictions added to dataframe!")
print(f"Added columns: {[f'DC_{task}' for task in tox21_tasks]}")
EnamineAntiviralsData_wToxicity_deepchem_final


 DeepChem-compatible predictions added to dataframe!
Added columns: ['DC_NR-AR', 'DC_NR-AR-LBD', 'DC_NR-AhR', 'DC_NR-Aromatase', 'DC_NR-ER', 'DC_NR-ER-LBD', 'DC_NR-PPAR-gamma', 'DC_SR-ARE', 'DC_SR-ATAD5', 'DC_SR-HSE', 'DC_SR-MMP', 'DC_SR-p53']


,Rank,SMILES,pPotency_prediction,IC50 (M),DC_NR-AR,DC_NR-AR-LBD,DC_NR-AhR,DC_NR-Aromatase,DC_NR-ER,DC_NR-ER-LBD,DC_NR-PPAR-gamma,DC_SR-ARE,DC_SR-ATAD5,DC_SR-HSE,DC_SR-MMP,DC_SR-p53
0,1,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,7.089,8.149146e-08,0.020714,0.010514,0.136735,0.023666,0.089574,0.037979,0.017535,0.141416,0.066652,0.045342,0.089071,0.052564
1,2,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,6.840,1.445621e-07,0.019247,0.010541,0.094948,0.034347,0.094115,0.023413,0.030239,0.125769,0.050595,0.037296,0.078907,0.037930
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,6.794,1.608349e-07,0.051570,0.033400,0.116305,0.025556,0.087873,0.019591,0.011712,0.108590,0.028621,0.065245,0.117644,0.039300
3,4,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,6.650,2.238847e-07,0.041308,0.021898,0.163825,0.033119,0.110762,0.024316,0.028378,0.132531,0.095349,0.045216,0.091601,0.052118
4,5,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,6.643,2.273610e-07,0.017548,0.012191,0.186113,0.061238,0.098721,0.051221,0.015372,0.103082,0.040477,0.057636,0.124210,0.048687
5,6,CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC,6.629,2.347164e-07,0.036747,0.016545,0.162314,0.027375,0.090101,0.025017,0.023862,0.135262,0.086518,0.038822,0.106418,0.089232
6,7,CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C,6.600,2.510905e-07,0.021897,0.022932,0.123533,0.036725,0.071555,0.025023,0.014081,0.133738,0.140694,0.036534,0.114930,0.071995
7,8,CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O,6.573,2.674945e-07,0.033732,0.063781,0.101444,0.033634,0.086850,0.023653,0.055597,0.153114,0.045955,0.039446,0.095676,0.056154
8,9,CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C,6.547,2.837417e-07,0.026535,0.010501,0.145244,0.029177,0.074724,0.022400,0.018525,0.105106,0.024037,0.035759,0.100122,0.080961
9,10,COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2,6.529,2.959537e-07,0.037943,0.025259,0.133014,0.044183,0.079210,0.020209,0.049972,0.100834,0.021206,0.031546,0.082406,0.058672


In [19]:
# Define critical vs less critical toxicity endpoints
critical_tox = ['DC_SR-p53', 'DC_NR-AR', 'DC_NR-ER']  # DNA damage, hormone disruption
moderate_tox = ['DC_NR-AhR', 'DC_SR-ARE', 'DC_SR-MMP']
all_tox = ['DC_NR-AR', 'DC_NR-AR-LBD', 'DC_NR-AhR', 'DC_NR-Aromatase', 
           'DC_NR-ER', 'DC_NR-ER-LBD', 'DC_NR-PPAR-gamma', 'DC_SR-ARE', 
           'DC_SR-ATAD5', 'DC_SR-HSE', 'DC_SR-MMP', 'DC_SR-p53']

# Calculate weighted toxicity score
df = EnamineAntiviralsData_wToxicity_deepchem_final.copy()

# Critical endpoints (weight 2x)
df['critical_tox_score'] = df[critical_tox].mean(axis=1)

# All endpoints
df['overall_tox_score'] = df[all_tox].mean(axis=1)

# Count critical violations (> 0.7 threshold)
df['critical_violations'] = (df[critical_tox] > 0.7).sum(axis=1)

# Count any violations (> 0.5 threshold)
df['total_violations'] = (df[all_tox] > 0.5).sum(axis=1)

# Potency score (convert IC50 to µM and invert)
df['IC50_uM'] = df['IC50 (M)'] * 1e6
df['potency_score'] = np.log10(1 / df['IC50_uM'])  # Log scale for potency

# Normalize scores to 0-1
df['potency_norm'] = (df['potency_score'] - df['potency_score'].min()) / \
                     (df['potency_score'].max() - df['potency_score'].min())
df['safety_norm'] = 1 - df['overall_tox_score']

# Final composite score
df['final_score'] = (
    0.5 * df['potency_norm'] +           # 50% potency
    0.35 * df['safety_norm'] +           # 35% overall safety
    0.15 * (1 - df['critical_tox_score'])  # 15% critical endpoints
)

# Penalize molecules with critical violations
df.loc[df['critical_violations'] > 0, 'final_score'] *= 0.5

# Rank
ranked = df.sort_values('final_score', ascending=False)

print("\n=== TOP 10 DRUG CANDIDATES ===")
print(ranked[['Rank', 'SMILES', 'pPotency_prediction', 'IC50 (M)', 'overall_tox_score', 'critical_violations', 
              'total_violations', 'final_score']].head(10).to_string(index=False))



=== TOP 10 DRUG CANDIDATES ===
 Rank                                          SMILES  pPotency_prediction     IC50 (M)  overall_tox_score  critical_violations  total_violations  final_score
    1  COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3                7.089 8.149146e-08           0.060980                    0                 0     0.970514
    2   COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O                6.840 1.445621e-07           0.053112                    0                 0     0.814717
    3      COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C                6.794 1.608349e-07           0.058784                    0                 0     0.781747
    4   COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3                6.650 2.238847e-07           0.070035                    0                 0     0.684717
    5    COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32                6.643 2.273610e-07           0.068041                    0                 0     0.683099
    6 CCC=1C=C

2.2 Use DeepChem's pre-trained Tox21 models